# Risk-to-Code Traceability

This notebook links code artifacts extracted by [Graphify](../../graphify) to AI risks from the
[AI Atlas Nexus](https://github.com/IBM/ai-atlas-nexus) risk ontology, and stores the result as an
RDF-star graph in an embedded [pyoxigraph](https://github.com/oxigraph/oxigraph) store.

Three passes generate each `(artifact, risk)` link, each cheaper and more precise than the last:

1. **Keywords** — a reverse index over risk `name`/`id` tokens shortlists candidates in O(1) per token.
2. **Embeddings** — `txtai` cosine similarity re-ranks the shortlist and surfaces risks Pass A missed.
3. **LLM verification** — Claude judges the top candidates and assigns a status: `confirmed`, `proposed` (needs human review), or `rejected`.

Every link carries its confidence, detection method, and rationale as RDF-star provenance, so the
graph stays auditable rather than a black box.

In [ ]:
#%pip install -e .. -q

import pandas as pd

from risk_code_traceability import run
from risk_code_traceability.store.queries import (
    community_risk_heatmap,
    high_confidence_proposed,
    risks_by_file,
)

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [7 lines of output]
      Checking for Rust toolchain....
      error: the 'cargo' binary, normally provided by the 'cargo' component, is not applicable to the 'stable-x86_64-apple-darwin' toolchain
      
      Cargo, the Rust package manager, is not installed or is not on PATH.
      This package requires Rust and Cargo to compile extensions. Install it through
      the system's package manager or via https://rustup.rs/
      
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> litellm

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Set skip_pass_c=False (and an ANTHROPIC_API_KEY env var) to run the full pipeline.
# skip_pass_c=True runs Pass A + Pass B only, so the notebook works without an API key.
result = run("../data/sample_graph.json", skip_pass_c=True)

In [ ]:
for key, value in result["stats"].items():
    print(f"{key:>17}: {value}")

In [ ]:
# Risks linked to a specific file
pd.DataFrame(risks_by_file(result["store"], "src/inference.py"))

In [ ]:
# Links above the confidence threshold that still need a human reviewer to confirm or reject them.
# `status` here is always "proposed" -- that is the human-review gate this query exists to surface.
pd.DataFrame(high_confidence_proposed(result["store"], threshold=0.5))

In [ ]:
# Confirmed risk links per Graphify community -- which parts of the codebase carry the most risk surface area.
pd.DataFrame(community_risk_heatmap(result["store"]))

In [ ]:
# The store is a standard SPARQL-star endpoint, not a black box -- any raw query works directly.
# Note the `<<( s p o )>>` quoted-triple syntax (parens required by this pyoxigraph version).
query = """
PREFIX graphify: <https://example.org/graphify-bridge/>
PREFIX atlas:    <https://ibm.github.io/ai-atlas-nexus/ontology/>
SELECT ?artifact ?status ?confidence WHERE {
    ?link graphify:asserts <<( ?artifact atlas:hasRelatedRisk ?risk )>> ;
          graphify:status ?status ;
          graphify:confidence ?confidence .
}
"""
for row in result["store"].query(query):
    print(row["artifact"], row["status"], row["confidence"])

## Going to production: swapping pyoxigraph for Neo4j

pyoxigraph is an embedded, zero-ops store well suited to a notebook demo. At production scale, the
same triples can be loaded into Neo4j instead: export the store as Turtle (`serialize_store`), convert
each RDF-star provenance quad into a relationship property using Atlas Nexus's existing Cypher export
tooling, and generate the load statements with the [`cymple`](https://github.com/koffiedev/cymple)
query builder. The bridge schema and query semantics stay identical -- only the storage backend changes.